# 05 · 事件研究框架

## 研究设计

对每一次评级行动（下调 / 上调 / 展望调整），考察事件日前后
$[-20, +20]$ 个**交易日**内目标变量的异常变化：

$$AR_{i,t} = R_{i,t} - E[R_{i,t}], \qquad CAR_i = \sum_{t=-20}^{+20} AR_{i,t}$$

## 正常收益的估计

| 模型 | 估计式 | 使用条件 |
| --- | --- | --- |
| 市场模型 | $R_{i,t} = \alpha + \beta R_{m,t} + \varepsilon$ | 估计窗口 $[-120, -21]$ 有至少 10 个观测且有市场因子 |
| 均值调整 | $E[R] = \overline{R}_{\text{est}}$ | 无市场因子或市场因子无变异 |
| 数据不足 | — | 窗口内有效观测不足，返回 `model = "insufficient_data"` 并保留缺失 |

## 数据可得性（重要）

**主权利差**：EMBI 全球利差（J.P. Morgan）、Bloomberg 主权 CDS 均为付费终端数据，
本项目**不打包**。可用的公开替代：

- FRED：部分新兴市场的利差 / 国债收益率序列（需自备 `FRED_API_KEY`）
- IMF IFS / World Bank：月度汇率与利率
- 各国央行 / 财政部公开数据

**汇率**：FRED、各国央行、IMF IFS 均可获得。
**股指**：FRED（部分国家）、Yahoo Finance 等公开源。

> ⚠️ **本项目不使用伪造数据填充缺失的高频序列。**
> 当行情数据不可得时，`event_study()` 会返回 `status="insufficient_data"`
> 的结果对象并说明限制，框架保持完整、其余分析不受影响。
>
> 下面用**合成的高频价格序列**演示完整的计算链路；该序列由本 notebook 就地生成，
> 仅用于验证代码路径，**不构成任何实证证据**。

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name and not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.analysis.event_study import (
    abnormal_changes,
    cumulative_abnormal_changes,
    event_study,
    event_study_summary,
    prepare_event_windows,
    ratings_to_events,
)
from src.config import get_path

EVENT_WINDOW = (-20, 20)
ESTIMATION_WINDOW = (-120, -21)

## 1. 从评级面板构造事件表

事件 = 实际发生评级行动的观测（`rating_change` 非空且方向匹配）。

In [ ]:
PANEL_PATH = get_path("processed") / "panel_country_year.csv"
if PANEL_PATH.exists():
    panel = pd.read_csv(PANEL_PATH, encoding="utf-8-sig", parse_dates=["action_date"])
else:
    from src.clean.panel import build_country_year_panel
    from src.ingest.ratings import load_sample_ratings

    panel = build_country_year_panel(
        load_sample_ratings(),
        pd.read_csv(get_path("sample") / "macro_sample.csv", encoding="utf-8-sig"),
        start_year=2000,
        end_year=2023,
    )

events = ratings_to_events(panel, event_types=("downgrade", "upgrade"))
print(f"事件数: {len(events):,}")
print(events.head(8).to_string(index=False))
print("\n事件类型分布:")
print(events["event_type"].value_counts())

## 2. 行情数据：先看「没有数据」时框架如何优雅降级

In [ ]:
# 不提供任何行情数据 —— 框架应返回明确的状态说明，而不是报错或编造数据
result_no_data = event_study(None, events)
print("状态:", result_no_data.status)
print("说明:", result_no_data.message)
print("元信息:", result_no_data.to_dict()["meta"])

## 3. 合成行情序列（仅用于验证代码路径）

真实研究中，此处应替换为「利差 / 汇率 / 股指」的**真实日频序列**，
长表列：`country_iso3`、`date`、`value`（可选 `market` 作为市场因子）。

下面生成的序列嵌入了「下调前 10 日利差走阔」的**人为效应**，
用于确认框架能正确识别已知效应。

In [ ]:
def make_synthetic_prices(events: pd.DataFrame, seed: int = 7) -> pd.DataFrame:
    """生成合成的日频价格序列（演示专用，不代表任何真实市场数据）。"""
    rng = np.random.default_rng(seed)
    frames = []
    for country in sorted(events["country_iso3"].unique()):
        start = pd.Timestamp("1999-01-01")
        dates = pd.bdate_range(start, "2024-12-31")
        market = rng.normal(0.0002, 0.011, len(dates))
        # 与市场因子相关的个股收益 + 特质噪声
        return_series = 0.7 * market + rng.normal(0.0001, 0.009, len(dates))
        country_events = events[events["country_iso3"] == country]
        for _, event in country_events.iterrows():
            event_date = pd.Timestamp(event["event_date"])
            pos = int(np.searchsorted(dates.to_numpy(), event_date.to_numpy()))
            if event["event_type"] == "downgrade" and pos > 12:
                # 下调事件前 10 个交易日人为加入累计 -2% 的异常收益
                return_series[max(pos - 10, 0) : pos] -= 0.002
        price = 100.0 * np.exp(np.cumsum(return_series))
        frames.append(
            pd.DataFrame({
                "country_iso3": country,
                "date": dates,
                "value": price,
                "market": 100.0 * np.exp(np.cumsum(market)),
            })
        )
    return pd.concat(frames, ignore_index=True)


PRICES = make_synthetic_prices(events)
print(f"合成行情: {len(PRICES):,} 行 | 国家 {PRICES['country_iso3'].nunique()}")
PRICES.head()

## 4. 构造事件窗口

In [ ]:
windows = prepare_event_windows(PRICES, events, window=EVENT_WINDOW)
print(f"事件窗口长表: {len(windows):,} 行 | 覆盖事件 {windows['event_id'].nunique()}")
print("每个事件的窗口长度分布:")
print(windows.groupby("event_id").size().describe().round(1))

## 5. 计算异常变化与 CAR

In [ ]:
abnormal = abnormal_changes(windows, estimation_window=ESTIMATION_WINDOW)
print("使用的正常收益模型分布:")
print(abnormal["model"].value_counts())

car = cumulative_abnormal_changes(abnormal, start=EVENT_WINDOW[0], end=EVENT_WINDOW[1])
summary = event_study_summary(abnormal, window=EVENT_WINDOW)
summary.round(5).head(15)

In [ ]:
# 按事件类型分别汇总：下调 vs 上调的 CAAR 形态应显著不同
merged = abnormal.merge(
    events[["country_iso3", "event_date", "event_type"]],
    on=["country_iso3", "event_date"],
    how="left",
)
by_type = (
    merged.dropna(subset=["abnormal"])
    .groupby(["event_type", "relative_day"])["abnormal"]
    .mean()
    .unstack("event_type")
    .cumsum()
)
by_type.round(5).loc[[-20, -15, -10, -5, -1, 0, 1, 5, 10, 20]]

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
for column in by_type.columns:
    fig.add_trace(go.Scatter(x=by_type.index, y=by_type[column], mode="lines", name=str(column)))
fig.add_vline(x=0, line_dash="dash", line_color="grey", annotation_text="事件日")
fig.update_layout(
    title="累积平均异常收益（CAAR）— 合成演示数据",
    xaxis_title="相对事件日（交易日）",
    yaxis_title="CAAR",
    template="plotly_white",
    height=440,
)
fig

## 6. 落地到真实数据的检查清单

1. **事件日期**：确认 `event_date` 是评级行动的**首次公开日**（而非报告发布日）。
2. **停牌/跳空**：节假日与停牌会破坏「相对交易日」对齐，需用交易日历重建。
3. **重叠事件**：同一国家短期内多次评级行动会导致窗口重叠，应剔除或加权。
4. **估计窗口污染**：若估计窗口内已有其他评级事件，市场模型会被污染，
   可改用更早的窗口或剔除该事件。
5. **统计推断**：CAR 的横截面相关性很强（同一时期多国被下调），
   需用聚类或 Boehmer–Musumeci–Poulsen (BMP) 标准化检验，而非简单 t 检验。
6. **预期效应**：评级行动常被市场提前预期，CAR 的「显著性」可能主要来自
   事件前的泄漏（本框架的相对日横截面正好可以检验这一点）。